# 核心问题：TP 的瓶颈在哪里？
## TP 的基本原理回顾
张量并行将模型参数（主要是 Linear 层的<font color='red'>权重矩阵</font>）按 特征维度（hidden dimension） 拆分：

In [ ]:
Column Parallel Linear:
  输入 X: [batch, seq, hidden]       (完整复制到所有 ranks)
  权重 W: [hidden, 4*hidden] → 拆分为 W0, W1 各 [hidden, 2*hidden]
  
  Rank 0: Y0 = X @ W0   → [batch, seq, 2*hidden]
  Rank 1: Y1 = X @ W1   → [batch, seq, 2*hidden]
  
  输出 Y = [Y0 | Y1]    → [batch, seq, 4*hidden] (列维度分片)

## TP 的显存瓶颈

In [ ]:
Transformer Block 的激活显存分布：
┌─────────────────────────────────────────┐
│  Attention 部分                          │
│   - QKV Linear 输入: [b, s, h] × 3       │  ← 完整复制到所有 TP ranks
│   - Attention Score: [b, s, s]           │  ← 完整复制
│   - Softmax 输出: [b, s, s]              │  ← 完整复制
├─────────────────────────────────────────┤
│  FFN 部分                                │
│   - LayerNorm 输入: [b, s, h]            │  ← 完整复制
│   - 第一个 Linear 输入: [b, s, h]        │  ← 完整复制
│   - GELU 输入/输出: [b, s, 4h]           │  ← 完整复制
│   - 第二个 Linear 输入: [b, s, 4h]       │  ← 完整复制
│   - Dropout mask: [b, s, h]            │  ← 完整复制
└─────────────────────────────────────────┘

### 关键问题：
- <font color='red'>TP 只拆分了 Linear 层的权重和输出，但 激活值（activations）在 LayerNorm、Dropout、Attention 等层是完整复制到所有 ranks 的。</font>

<font color='blue'>当序列长度 s  很大时（如 32K, 100K），这些完整复制的激活成为显存瓶颈。</font>


# 序列并行（SP）的引入
## 核心思想
- 既然 LayerNorm、Dropout、GELU 等操作是 token-wise 的（<font color='red'>每个 token 独立计算</font>），为什么不让它们在序列维度分片？

In [ ]:
标准 TP（仅特征维度拆分）:
┌─────────┐     ┌─────────┐
│ Rank 0  │     │ Rank 1  │
│ [b,s,h] │     │ [b,s,h] │   ← 输入完整复制
│   ↓     │     │   ↓     │
│ LayerNorm│    │ LayerNorm│   ← 各算各的，结果一样（浪费！）
│ [b,s,h] │     │ [b,s,h] │
│   ↓     │     │   ↓     │
│ Linear  │     │ Linear  │   ← 这里才用到 TP
│ [b,s,2h]│     │ [b,s,2h]│
└─────────┘     └─────────┘

序列并行 SP（序列维度也拆分）:
┌─────────┐     ┌─────────┐
│ Rank 0  │     │ Rank 1  │
│[b,s/2,h]│     │[b,s/2,h]│   ← 输入按序列分片
│   ↓     │     │   ↓     │
│ LayerNorm│    │ LayerNorm│   ← 各算各的，但算的是不同 token！
│[b,s/2,h]│     │[b,s/2,h]│
│   ↓     │     │   ↓     │
│ All-Gather│   │ All-Gather│  ← 恢复完整序列，供 Linear 使用
│ [b,s,h] │     │ [b,s,h] │
│   ↓     │     │   ↓     │
│ Linear  │     │ Linear  │
│ [b,s,2h]│     │ [b,s,2h]│
└─────────┘     └─────────┘

# 详细对比：TP 单独 vs TP+SP 联合
## 例子：GPT-3 风格的 Transformer Block
### 超参数：
- Batch size b=2 
- Sequence length s=4096 
- Hidden dim h=4096 
- FFN expansion = 4（中间维度 4h=16384 ）
- TP size = 2（2 张 GPU）

## 场景 A：仅使用 TP（无 SP）

In [ ]:
输入 X: [2, 4096, 4096]

┌─────────────────────────────────────────────────────────────┐
│ 1. Pre-Attention LayerNorm                                   │
│    输入: [2, 4096, 4096] 完整复制到 Rank 0 和 Rank 1          │
│    每个 rank 显存: 2 × 4096 × 4096 × 2B (fp16) = 64 MB       │
│    总显存: 64 × 2 = 128 MB                                   │
├─────────────────────────────────────────────────────────────┤
│ 2. QKV Linear (Column Parallel)                              │
│    输入: [2, 4096, 4096] 完整复制                            │
│    权重: W_qkv [4096, 12288] → 拆为 [4096, 6144] 每 rank     │
│    输出: [2, 4096, 6144] 列分片                              │
├─────────────────────────────────────────────────────────────┤
│ 3. Attention Score QK^T                                      │
│    Q, K: [2, 4096, 4096] 完整复制（为了计算完整的 attention）  │
│    Score: [2, 4096, 4096] 完整复制                            │
│    显存: 2 × 4096 × 4096 × 2B × 3 = 192 MB 每 rank           │
├─────────────────────────────────────────────────────────────┤
│ 4. Softmax                                                   │
│    输入/输出: [2, 4096, 4096] 完整复制                        │
│    显存: 128 MB 每 rank                                      │
├─────────────────────────────────────────────────────────────┤
│ 5. Attention Output Projection                               │
│    输入: [2, 4096, 4096] 完整复制                            │
│    输出: [2, 4096, 4096] 行并行，需要 All-Reduce             │
├─────────────────────────────────────────────────────────────┤
│ 6. Dropout (Post-Attention)                                  │
│    mask: [2, 4096, 4096] 完整复制                            │
│    显存: 64 MB 每 rank                                       │
├─────────────────────────────────────────────────────────────┤
│ 7. Pre-FFN LayerNorm                                         │
│    输入: [2, 4096, 4096] 完整复制                            │
│    显存: 64 MB 每 rank                                       │
├─────────────────────────────────────────────────────────────┤
│ 8. FFN 第一个 Linear (Column Parallel)                       │
│    输入: [2, 4096, 4096] 完整复制                            │
│    输出: [2, 4096, 8192] 列分片 (16384/2)                    │
├─────────────────────────────────────────────────────────────┤
│ 9. GELU Activation                                           │
│    输入/输出: [2, 4096, 8192] 列分片                         │
│    显存: 128 MB 每 rank                                      │
├─────────────────────────────────────────────────────────────┤
│ 10. FFN 第二个 Linear (Row Parallel)                         │
│    输入: [2, 4096, 8192] 列分片                              │
│    输出: [2, 4096, 4096] 需要 All-Reduce 后完整复制           │
├─────────────────────────────────────────────────────────────┤
│ 11. Dropout (Post-FFN)                                       │
│    mask: [2, 4096, 4096] 完整复制                            │
│    显存: 64 MB 每 rank                                       │
└─────────────────────────────────────────────────────────────┘

LayerNorm + Dropout 相关激活显存（每 rank）:
  = 64 + 64 + 64 + 64 = 256 MB

当 s = 32768 时:
  = 256 MB × (32768/4096)² = 256 × 64 = 16,384 MB = 16 GB 每 rank！

## 场景 B：TP + SP 联合使用

In [ ]:
输入 X: [2, 4096, 4096]

按序列维度分片（SP）:
  Rank 0: X0 = X[:, 0:2048, :]    → [2, 2048, 4096]
  Rank 1: X1 = X[:, 2048:4096, :] → [2, 2048, 4096]

┌─────────────────────────────────────────────────────────────┐
│ 1. Pre-Attention LayerNorm (SP 生效！)                      │
│    Rank 0 输入: [2, 2048, 4096]                              │
│    Rank 1 输入: [2, 2048, 4096]                              │
│    每个 rank 显存: 2 × 2048 × 4096 × 2B = 32 MB              │
│    ✅ 节省一半！                                              │
├─────────────────────────────────────────────────────────────┤
│ 2. QKV Linear (需要 All-Gather 恢复完整序列)                  │
│    All-Gather: [2, 2048, 4096] → [2, 4096, 4096]             │
│    然后正常 Column Parallel Linear...                        │
├─────────────────────────────────────────────────────────────┤
│ 3. Attention Score (Attention 需要完整序列)                   │
│    Q, K, V: [2, 4096, 4096] 完整复制                         │
│    这里 SP 不生效，需要 Ring Attention 等特殊处理              │
├─────────────────────────────────────────────────────────────┤
│ 4. Softmax, Output Projection...                            │
│    （Attention 内部保持完整序列）                              │
├─────────────────────────────────────────────────────────────┤
│ 5. Dropout (Post-Attention) (SP 生效！)                      │
│    输出按序列分片: [2, 2048, 4096] 每 rank                     │
│    显存: 32 MB 每 rank ✅ 节省一半                            │
├─────────────────────────────────────────────────────────────┤
│ 6. Pre-FFN LayerNorm (SP 生效！)                             │
│    输入: [2, 2048, 4096] 每 rank                              │
│    显存: 32 MB ✅ 节省一半                                    │
├─────────────────────────────────────────────────────────────┤
│ 7. FFN 第一个 Linear (需要 All-Gather)                       │
│    All-Gather 恢复 [2, 4096, 4096]                           │
│    Column Parallel → 输出 [2, 4096, 8192] 列分片              │
├─────────────────────────────────────────────────────────────┤
│ 8. GELU (SP 生效！)                                          │
│    输入是列分片的，但可以在序列维度进一步分片吗？              │
│    实际上 GELU 是 element-wise，如果输入已列分片，            │
│    输出也是列分片。但如果要序列分片，需要额外的通信...         │
│    在 Megatron 中，FFN 中间态通常保持列分片，不额外序列分片   │
├─────────────────────────────────────────────────────────────┤
│ 9. FFN 第二个 Linear (Row Parallel)                          │
│    输入: [2, 4096, 8192] 列分片                              │
│    输出: All-Reduce 后 [2, 4096, 4096]                       │
├─────────────────────────────────────────────────────────────┤
│ 10. Dropout (Post-FFN) (SP 生效！)                          │
│    按序列分片: [2, 2048, 4096] 每 rank                        │
│    显存: 32 MB ✅ 节省一半                                    │
└─────────────────────────────────────────────────────────────┘

LayerNorm + Dropout 相关激活显存（每 rank）:
  = 32 + 32 + 32 + 32 = 128 MB（对比纯 TP 的 256 MB，节省一半）

当 s = 32768, TP=8, SP=8 时:
  序列每段长度 = 32768 / 8 = 4096
  LayerNorm 显存 = 2 × 4096 × 4096 × 2B = 64 MB（每 rank）
  对比无 SP: 2 × 32768 × 4096 × 2B = 512 MB
  节省 8 倍！

# TP 和 SP 的互补关系图解

In [ ]:
┌─────────────────────────────────────────────────────────────────┐
│                    Transformer Block                             │
│                                                                  │
│   Input: [b, s, h]                                               │
│      │                                                           │
│      ▼                                                           │
│   ┌─────────────┐     ┌─────────────┐     ┌─────────────┐       │
│   │  LayerNorm  │────▶│   Attention │────▶│   Dropout   │       │
│   │   (SP ✅)   │     │  (SP ❌)    │     │   (SP ✅)   │       │
│   └─────────────┘     └─────────────┘     └─────────────┘       │
│         ↑                                      │                 │
│         │                                      ▼                 │
│         │                              ┌─────────────┐         │
│         │                              │  Residual   │         │
│         │                              │   Add       │         │
│         │                              └─────────────┘         │
│         │                                      │                 │
│         │                                      ▼                 │
│         │                              ┌─────────────┐         │
│         └──────────────────────────────│  LayerNorm  │         │
│                                        │   (SP ✅)   │         │
│                                        └─────────────┘         │
│                                              │                   │
│                                              ▼                   │
│                                        ┌─────────────┐         │
│                                        │ Linear (TP) │         │
│                                        │  [h→4h]     │         │
│                                        └─────────────┘         │
│                                              │                   │
│                                              ▼                   │
│                                        ┌─────────────┐         │
│                                        │    GELU     │         │
│                                        │   (SP ✅)   │         │
│                                        └─────────────┘         │
│                                              │                   │
│                                              ▼                   │
│                                        ┌─────────────┐         │
│                                        │ Linear (TP) │         │
│                                        │  [4h→h]     │         │
│                                        └─────────────┘         │
│                                              │                   │
│                                              ▼                   │
│                                        ┌─────────────┐         │
│                                        │   Dropout   │         │
│                                        │   (SP ✅)   │         │
│                                        └─────────────┘         │
│                                              │                   │
│                                              ▼                   │
│                                        ┌─────────────┐         │
│                                        │  Residual   │         │
│                                        │   Add       │         │
│                                        └─────────────┘         │
│                                              │                   │
│                                           Output                 │
└─────────────────────────────────────────────────────────────────┘

图例:
  (SP ✅) = 序列并行生效，激活按序列分片，节省显存
  (SP ❌) = 序列并行不生效，需要完整序列（Attention 的 Softmax）
  (TP)   = 张量并行生效，权重按特征分片

# 通信模式对比
## 纯 TP 的通信

In [ ]:
每层的通信:
1. Attention Output Projection: All-Reduce (Row Parallel)
2. FFN 第二个 Linear: All-Reduce (Row Parallel)

通信量: 2 × [b, s, h] 每 layer

## TP + SP 的通信

In [ ]:
每层的通信:
1. Pre-Attention: All-Gather (SP → TP 切换)
   序列分片 [b, s/tp, h] → 完整 [b, s, h]
   
2. Attention Output Projection: All-Reduce (Row Parallel)
   [b, s, h] → 完整 [b, s, h]
   
3. Post-Attention: Reduce-Scatter (TP → SP 切换)
   完整 [b, s, h] → 序列分片 [b, s/tp, h]
   
4. Pre-FFN: 已经是序列分片，不需要通信
   
5. FFN 第一个 Linear 前: All-Gather (SP → TP 切换)
   序列分片 [b, s/tp, h] → 完整 [b, s, h]
   
6. FFN 第二个 Linear: All-Reduce (Row Parallel)
   [b, s, h] → 完整 [b, s, h]
   
7. Post-FFN: Reduce-Scatter (TP → SP 切换)
   完整 [b, s, h] → 序列分片 [b, s/tp, h]

额外通信: 2 × All-Gather + 2 × Reduce-Scatter 每 layer

### 关键洞察：
- 额外通信量 ≈ 4 × [b, s/tp, h]（因为 All-Gather 和 Reduce-Scatter 的数据量是序列分片后的大小）
- 但节省的显存 ≈ 多个 [b, s, h]（LayerNorm、Dropout 等）
- 当序列很长时，显存节省的收益 > 额外通信的开销


# Megatron-LM 中的实际实现

In [ ]:
# Megatron-LM 的核心代码逻辑（简化版）

class ParallelTransformerLayer:
    def __init__(self, config):
        self.sequence_parallel = config.sequence_parallel  # 是否启用 SP
        
        # LayerNorm: 可以在序列维度分片
        self.input_layernorm = LayerNorm(config.hidden_size)
        
        # Attention: 需要完整序列（或 Ring Attention）
        self.self_attention = ParallelAttention(config)
        
        # MLP: Linear 需要完整序列输入
        self.mlp = ParallelMLP(config)
    
    def forward(self, hidden_states):
        # hidden_states 的形状取决于是否启用 SP
        
        # ========== Attention 分支 ==========
        
        # 1. LayerNorm: 如果是 SP 模式，输入是序列分片的
        #    不需要通信，每个 token 独立计算
        layernorm_output = self.input_layernorm(hidden_states)
        
        # 2. 如果启用 SP，需要 All-Gather 恢复完整序列，供 Attention 使用
        if self.sequence_parallel:
            # All-Gather: [b, s/tp, h] → [b, s, h]
            layernorm_output = all_gather(layernorm_output, dim=1)
        
        # 3. Attention: 需要完整序列（Q, K, V 的交互）
        attention_output = self.self_attention(layernorm_output)
        
        # 4. Attention 输出是完整序列，需要 Reduce-Scatter 回序列分片
        if self.sequence_parallel:
            # Reduce-Scatter: [b, s, h] → [b, s/tp, h]
            attention_output = reduce_scatter(attention_output, dim=1)
        
        # 5. Dropout: 序列分片，不需要通信
        hidden_states = hidden_states + self.dropout(attention_output)
        
        # ========== FFN 分支 ==========
        
        # 6. LayerNorm: 序列分片，不需要通信
        layernorm_output = self.post_attention_layernorm(hidden_states)
        
        # 7. 如果启用 SP，All-Gather 恢复完整序列
        if self.sequence_parallel:
            layernorm_output = all_gather(layernorm_output, dim=1)
        
        # 8. MLP: Linear 需要完整序列
        mlp_output = self.mlp(layernorm_output)
        
        # 9. MLP 输出 Reduce-Scatter 回序列分片
        if self.sequence_parallel:
            mlp_output = reduce_scatter(mlp_output, dim=1)
        
        # 10. Dropout: 序列分片
        output = hidden_states + self.dropout(mlp_output)
        
        return output

# 为什么 SP 是 TP 的"补充"而非"替代"？

| 维度          | 张量并行 (TP)               | 序列并行 (SP)                   |
| ----------- | ----------------------- | --------------------------- |
| **拆分维度**    | 特征维度 (hidden dim)       | 序列维度 (seq len)              |
| **解决的问题**   | 模型参数量太大，单卡放不下           | 激活显存太大，长序列放不下               |
| **适用层**     | Linear 层（矩阵乘法）          | LayerNorm, Dropout, GELU    |
| **不适用层**    | —                       | Attention（需要全局 Softmax）     |
| **通信类型**    | All-Reduce / All-Gather | All-Gather / Reduce-Scatter |
| **扩展性限制**   | 受限于 hidden\_dim（通常 ≤8）  | 受限于 seq\_len（可以很大）          |
| **与 TP 关系** | 基础并行策略                  | **补充** TP，处理 TP 未覆盖的激活      |

## 互补性的本质

In [ ]:
TP 拆分的是"宽度"（特征维度）:
  模型太宽（hidden_dim=12288）→ TP=8，每卡 1536 维
  但 TP 无法拆分 LayerNorm/Dropout 的激活

SP 拆分的是"长度"（序列维度）:
  序列太长（seq_len=100K）→ SP=8，每卡 12.5K tokens
  专门处理 TP 未覆盖的激活显存

两者结合:
  TP=8 处理权重和 Linear 计算
  SP=8 处理 LayerNorm/Dropout 激活
  总并行度 = TP × SP = 64（需要 64 张 GPU）

# 实际收益：以 LLaMA-65B 为例

In [ ]:
模型配置:
  - Layers: 80
  - Hidden dim: 8192
  - FFN dim: 22016
  - Heads: 64
  - Seq len: 32768
  - Batch size: 1

仅 TP=8（无 SP）:
  - LayerNorm 激活: 80 × 2 × 32768 × 8192 × 2B ≈ 85 GB（每卡）
  - Dropout mask: 类似量级
  - 总激活显存 > 200 GB/卡 → 超出 A100 80GB 显存！

TP=8 + SP=8:
  - LayerNorm 激活: 85 GB / 8 = 10.6 GB（每卡）
  - Dropout mask: 类似减少
  - 总激活显存 < 50 GB/卡 → 可以放入 A100 80GB！

代价:
  - 额外通信: 每层 4 次集合通信（All-Gather + Reduce-Scatter）
  - 但通信量与序列长度成正比，而计算量与序列长度平方成正比
  - 长序列时，通信开销占比 < 5%

# 总结
序列并行（SP）是对张量并行（TP）的补充，因为：
- TP 解决了"模型太宽"的问题（权重按特征维度拆分）
- 但 TP 留下了"激活太长"的问题（LayerNorm/Dropout 的激活仍完整复制）
- SP 专门解决"激活太长"的问题（将这些激活按序列维度拆分）
- 两者结合：TP 处理 Linear 层的权重并行，SP 处理非 Linear 层的激活并行，实现全方位的显存优化

In [ ]:
┌─────────────────────────────────────────┐
│           完整的并行策略栈               │
├─────────────────────────────────────────┤
│  Data Parallel (DP)      → 批次拆分      │
│      ↓                                  │
│  Tensor Parallel (TP)    → 特征拆分      │
│      ↓                                  │
│  Sequence Parallel (SP)  → 序列拆分      │  ← TP 的补充！
│      ↓                                  │
│  Pipeline Parallel (PP)  → 层间拆分      │
└─────────────────────────────────────────┘